# Video foundation model feature extraction 


For now, the extraction is performed using the VideoMAEv2, without fine-tuning on our video data.


Running this notebook should resul in launching the extraction of video features for all the Smartflat dataset. 

# Definition of the notations 

- $l$ : segment_length, number of frames that are used by the video foundation model to compute the representation
- $\Delta T$: delta_t, elapsed number of frames between each segments


In [ ]:
"""Extract features for temporal action detection datasets

Sam Perochon, adapted from https://github.com/OpenGVLab/VideoMAEv2. December 2023.

"""
%load_ext autoreload
%autoreload 2
from main import *
import time
import socket
import logging
from  tqdm import tqdm
import os
# Si deux GPUS: 

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

logging.basicConfig(filename=os.path.join(get_data_root(), 'log','video_representations_computation.log'),
                    filemode='a',
                    format='%(asctime)s,%(msecs)d %(name)s %(levelname)s %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S',
                    level=logging.DEBUG)

logging.info("Video Foundation Model Computation")
logger = logging.getLogger('main')

# Define model and video dataset path
path = os.path.join(get_data_root(), 'dataframes', 'persistent_metadata', '{}_dataset_df.csv'.format(socket.gethostname()))
model_ckpt = os.path.join(os.getcwd(), 'models/vit_g_hybrid_pt_1200e_k710_ft.pth')
model_name = 'vit_giant_patch14_224'
metrics_path = os.path.join(get_data_root(), 'dataframes', 'compute_time_video_representation_learning.csv')

segment_length = 16 # Frame per segments
delta_t = 8 # Elapsed number of frames between each temporal segments being represented
logging_interval = 2000

# Fecth video path to process    
video_paths = fetch_video_paths(path, model_name, shuffle=False)

# Get video loader 
video_loader = get_video_loader()
start_idx_range = smartflat_range
transform = transforms.Compose([ToFloatTensorInZeroOne(),Resize((224, 224))])

# Get model
model = get_model(model_name, model_ckpt)
model.to(device)
None

In [ ]:
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))))
from api.datasets.loader import get_dataset

dset = get_dataset(dataset_name='video_block_representation', root_dir=get_data_root(), scenario='all')

In [ ]:
# Get list of video to process
df = pd.read_csv(path).sort_values(['task_name', 'participant_id'])
df['has_collate_video'] = df.apply(lambda x: os.path.isfile(os.path.join(get_data_root(), x.folder, x.participant_id, x.modality, 'merged_video.mp4')), axis=1)

df_filtered = df[(~df['video_path'].isna()) & 
                    (df['video_representation_computed'] == False) &  
                    (df['flag_video_representation'] != 'failure') &  
                    ((df['video_name'] == 'merged_video') | (df['n_videos'] == 1))]

video_paths = df_filtered['video_path'].dropna().unique()
#video_paths = [video_path for video_path in video_paths if not os.path.exists(fetch_output_path(video_path, model_name))]
print(len(video_paths))
video_paths

In [ ]:
metrics = [] 
n = 0
while n <1:

    # Fecth video path to process    
    #video_paths = fetch_video_paths(path, model_name, shuffle=False)

    # extract feature
    for idx, video_path in enumerate(video_paths):

        start_time = time.time()
        print("Extracting feature {}/{} for {}".format(idx+1, len(video_paths), video_path))
        
        output_file = fetch_output_path(video_path, model_name)
        
        if os.path.isfile(output_file):
            print("Already done.")
            continue
            
        try:
            vr = video_loader(video_path)
        except:            
            # failure flag
            logger.info("[FAILURE] Unreadable video {}".format(video_path))
            with open(os.path.join(os.path.dirname(output_file), f'.{os.path.basename(video_path)[:-4]}_video_representation_flag.txt'), 'w') as f:
                f.write('failure')
      
            continue

        if len(vr) <= segment_length:
            
            # failure flag
            logger.info("[FAILURE] Video duration does not meet minimum length {}".format(video_path))
            with open(os.path.join(os.path.dirname(output_file), f'.{os.path.basename(video_path)[:-4]}_video_representation_flag.txt'), 'w') as f:
                f.write('failure')
            continue
            
        
        
        print({'video_path': video_path, 'num_frames': len(vr), 'video_representations_compute_time': time.time() - start_time})
    
        # Extract feature
        feature_list = []
        for i, start_idx in enumerate(tqdm(start_idx_range(len(vr), segment_length, delta_t))):
            
            try:
                data = vr.get_batch(np.arange(start_idx, start_idx + segment_length)).asnumpy()
                frame = torch.from_numpy(data)  # torch.Size([16, 566, 320, 3])
                frame_q = transform(frame)  # torch.Size([3, 16, 224, 224])
                input_data = frame_q.unsqueeze(0).to(device)
    
                with torch.no_grad():
                    feature = model.forward_features(input_data)
                    feature_list.append(feature.cpu().numpy())
            except:
                feature_list.append(np.zeros((1, 1408)))
    

            if i % logging_interval == 0:
                logger.info("{}/{} {} - {}/{} [{:.2f} %]".format(idx, len(video_paths), video_path, i, len(start_idx_range(len(vr), segment_length, delta_t)), 100*i/len(start_idx_range(len(vr), segment_length, delta_t))))

        # Add check to see if all embeddings computation failed
        if np.vstack(feature_list).mean() == 0:    

            logger.info("[FAILURE] All block embeddings are null {}".format(video_path))
            with open(os.path.join(os.path.dirname(output_file), f'.{os.path.basename(video_path)[:-4]}_video_representation_flag.txt'), 'w') as f:
                f.write('failure')
                
        
        # [N, C]
        np.save(output_file, np.vstack(feature_list))
        metrics.append({'video_path': video_path, 'num_frames': len(vr), 'video_representations_compute_time': time.time() - start_time, 'segment_length': segment_length, 'delta_t': delta_t})
        print(f'[{output_file} / {len(video_paths)}]: saved feature on {output_file}')
        print({'video_path': video_path, 'num_frames': len(vr), 'video_representations_compute_time': time.time() - start_time})
        print("Elapsed time: {:.2f}s".format(time.time() - start_time))
        print('Finished video!')
        #except:
        #    print("Extracting feature {}/{} for {}".format(idx+1, len(video_paths), video_path))
        #    print("Failed.")
        
        # success flag
        with open(os.path.join(os.path.dirname(output_file), f'.{os.path.basename(video_path)[:-4]}_video_representation_flag.txt'), 'w') as f:
            f.write('success')
        print('success flag: {}'.format(os.path.join(os.path.dirname(output_file), f'.{os.path.basename(video_path)[:-4]}_video_representation_flag.txt')))
                
        
        # Save logging df
        if os.path.isfile(metrics_path):
            metricsdf = pd.concat([pd.read_csv(metrics_path).assign(delta_t=delta_t, segment_length=segment_length), pd.DataFrame(metrics)])
        else:
            metricsdf =  pd.DataFrame(metrics)
        metricsdf.to_csv(metrics_path, index=False)



    # Attendre 2 heures
    #time.sleep(2 * 60 * 60)
    n+=1 

sys.exit(0)

In [ ]:
np.load('/diskA/sam_data/data/cuisine/G45_P32_CARCat_17072018/GoPro1/video_representations_VideoMAEv2_merged_video.npy')